In [1]:
import numpy as np
from h5py import File
from matplotlib import pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader,Dataset
filename = 'feature.h5'

In [3]:
with File(filename,'r') as h5:
    print(h5.keys())
    
    src_label = h5['src_label'][:]
    src_feature = h5['src_feature'][:]

    
    tgt_label = h5['tgt_label'][:]
    tgt_feature = h5['tgt_feature'][:]

<KeysViewHDF5 ['src_feature', 'src_label', 'tgt_feature', 'tgt_label']>


In [5]:
def scl(p):
    p = np.clip(p,1e-8,1)
    p_mean = p.mean(axis=0)
    return -np.mean(p*np.log(p))+np.mean(p_mean*np.log(p_mean))
def scl_y(y):
   y_max = y.max(axis=-1,keepdims=True)
   dy = y_max-y
   return -dy.mean()
def scl_torch(p):
    p = torch.clip(p,1e-8,1)
    p_mean = p.mean(dim=0)
    return -torch.mean(p*torch.log(p))+torch.mean(p_mean*torch.log(p_mean))*10
def scl_torch_y(y):
   y_max = y.max(dim=-1,keepdims=True).values
   dy = (y_max-y)**2
   return -dy.mean()
def top1(y_P,label):
    label_P = y_P.argmax(axis=-1)
    r = np.where(label_P==label,1,0)
    return r.mean()

In [4]:
device = 'cuda:1'
class DY(torch.nn.Module):
    def __init__(self,class_num,feature_num):
        super().__init__()
        self. dy = torch.nn.Parameter(torch.zeros(class_num))
        self.M = torch.nn.Parameter(torch.zeros((class_num,feature_num)))
    def forward(self,f):
        return self.dy.unsqueeze(0)+torch.einsum('bf,cf->bc',f,self.M)

In [6]:
src_label_torch = torch.from_numpy(src_label).to(device)
src_feature_torch = torch.from_numpy(src_feature).to(device)
tgt_label_torch = torch.from_numpy(tgt_label).to(device)
tgt_feature_torch = torch.from_numpy(tgt_feature).to(device)

In [ ]:

dy = DY(31,2048)
dy.to(device)

w = 0.01
dy.train()
opt = torch.optim.SGD(dy.parameters(),lr=1e-2)
sclL = []
tgt_acL =[]
for i in range(10000):
    dy.zero_grad()
    tgt_y_torch_dy  = dy(tgt_feature_torch )
    tgt_y_torch_dy_p = torch.nn.functional.softmax(tgt_y_torch_dy,dim=-1)
    
    loss_sur = torch.nn.functional.cross_entropy(tgt_y_torch_dy,tgt_label_torch)
    loss = loss_sur

    loss.backward()
    opt.step()
    tgt_y_torch_dy_num = tgt_y_torch_dy.detach().cpu().numpy()
    tgt_ac = top1(tgt_y_torch_dy_num,tgt_label)
    if i%10==0:
        print(tgt_ac)
        #sclL.append(loss_scl.item())
        tgt_acL.append(tgt_ac)

0.032670454545454544
0.6814630681818182
0.7269176136363636
0.7503551136363636
0.7610085227272727
0.7727272727272727
0.7830255681818182
0.7926136363636364
0.7972301136363636
0.7993607954545454
0.8036221590909091
0.8075284090909091
0.8139204545454546
0.8178267045454546
0.8199573863636364
0.8238636363636364
0.8259943181818182
0.8270596590909091
0.8302556818181818
0.8313210227272727
0.8341619318181818
0.8345170454545454
0.8348721590909091
0.8355823863636364
0.8370028409090909
0.8373579545454546
0.8384232954545454
0.83984375
0.8405539772727273
0.8419744318181818
0.8430397727272727
0.84375
0.8451704545454546
0.8458806818181818
0.8462357954545454
0.8473011363636364
0.8487215909090909
0.8497869318181818
0.8497869318181818
0.8508522727272727
0.8512073863636364
0.8529829545454546
0.8536931818181818
0.8536931818181818
0.8583096590909091
0.8586647727272727
0.8597301136363636
0.8607954545454546
0.8618607954545454
0.86328125
0.8639914772727273
0.8647017045454546
0.8654119318181818
0.8664772727272727

In [18]:
dy = DY(31,2048)
dy.to(device)

w = 0.01
dy.train()
opt = torch.optim.SGD(dy.parameters(),lr=1e-2)
opt= torch.optim.AdamW(dy.parameters(),lr=1e-3)
sclL = []
tgt_acL =[]
T =1
for i in range(20000):
    dy.zero_grad()
    tgt_y_torch_dy  = dy(tgt_feature_torch )
    tgt_y_torch_dy_p = torch.nn.functional.softmax(tgt_y_torch_dy/T,dim=-1)
    
    src_y_torch_dy  = dy(src_feature_torch )
    src_y_torch_dy_p = torch.nn.functional.softmax(src_y_torch_dy/T,dim=-1)
    
    loss_sur = torch.nn.functional.cross_entropy(src_y_torch_dy,src_label_torch)
    loss_scl = scl_torch(tgt_y_torch_dy_p)
    #loss_sur = torch.nn.functional.cross_entropy(tgt_y_torch_dy,tgt_label_torch)
    loss =loss_sur#+(1-alpha)*loss_scl

    loss.backward()
    opt.step()
    tgt_y_torch_dy_num = tgt_y_torch_dy.detach().cpu().numpy()
    tgt_ac = top1(tgt_y_torch_dy_num,tgt_label)
    if i%10==0:
        print(tgt_ac)
        #sclL.append(loss_scl.item())
        tgt_acL.append(tgt_ac)

0.032670454545454544
0.5894886363636364
0.6470170454545454
0.6605113636363636
0.6580255681818182
0.6569602272727273
0.65625
0.6551846590909091
0.6541193181818182
0.6541193181818182
0.6541193181818182
0.6544744318181818
0.6548295454545454
0.6548295454545454
0.6551846590909091
0.6555397727272727
0.6558948863636364
0.65625
0.6566051136363636
0.6558948863636364
0.65625
0.65625
0.6558948863636364
0.65625
0.65625
0.65625
0.6558948863636364
0.6555397727272727
0.6551846590909091
0.6555397727272727
0.6555397727272727
0.6558948863636364
0.6558948863636364
0.6558948863636364
0.6551846590909091
0.6555397727272727
0.6558948863636364
0.6548295454545454
0.6544744318181818
0.6544744318181818
0.6544744318181818
0.6544744318181818
0.6544744318181818
0.6548295454545454
0.6548295454545454
0.6548295454545454
0.6548295454545454
0.6551846590909091
0.6548295454545454
0.6551846590909091
0.6551846590909091
0.6551846590909091
0.6555397727272727
0.6558948863636364
0.6558948863636364
0.6558948863636364
0.655894886

In [17]:
dy = DY(31,2048)
dy.to(device)

w = 0.01
dy.train()
opt = torch.optim.SGD(dy.parameters(),lr=1e-2)
opt= torch.optim.AdamW(dy.parameters(),lr=1e-3)
sclL = []
tgt_acL =[]
T =1
for i in range(20000):
    alpha = torch.exp(torch.tensor(-i/2000)).to(device)*0.95+0.05
    dy.zero_grad()
    tgt_y_torch_dy  = dy(tgt_feature_torch )
    tgt_y_torch_dy_p = torch.nn.functional.softmax(tgt_y_torch_dy/T,dim=-1)
    
    src_y_torch_dy  = dy(src_feature_torch )
    src_y_torch_dy_p = torch.nn.functional.softmax(src_y_torch_dy/T,dim=-1)
    
    loss_sur = torch.nn.functional.cross_entropy(src_y_torch_dy,src_label_torch)
    loss_scl = scl_torch(tgt_y_torch_dy_p)
    #loss_sur = torch.nn.functional.cross_entropy(tgt_y_torch_dy,tgt_label_torch)
    loss = alpha*loss_sur+(1-alpha)*loss_scl

    loss.backward()
    opt.step()
    tgt_y_torch_dy_num = tgt_y_torch_dy.detach().cpu().numpy()
    tgt_ac = top1(tgt_y_torch_dy_num,tgt_label)
    if i%10==0:
        print(tgt_ac)
        #sclL.append(loss_scl.item())
        tgt_acL.append(tgt_ac)

0.032670454545454544
0.5901988636363636
0.6473721590909091
0.6622869318181818


0.6594460227272727
0.6619318181818182
0.6633522727272727
0.6661931818181818
0.671875
0.6764914772727273
0.6786221590909091
0.6821732954545454
0.6857244318181818
0.6857244318181818
0.6871448863636364
0.6871448863636364
0.6889204545454546
0.6892755681818182
0.6899857954545454
0.6921164772727273
0.6921164772727273
0.6938920454545454
0.6928267045454546
0.6949573863636364
0.6949573863636364
0.6953125
0.6946022727272727
0.6956676136363636
0.6963778409090909
0.6967329545454546
0.6970880681818182
0.6974431818181818
0.6970880681818182
0.6974431818181818
0.6985085227272727
0.69921875
0.6995738636363636
0.7006392045454546
0.7017045454545454
0.7013494318181818
0.7017045454545454
0.7027698863636364
0.7034801136363636
0.7049005681818182
0.7049005681818182
0.7049005681818182
0.7049005681818182
0.7049005681818182
0.7063210227272727
0.70703125
0.7059659090909091
0.7059659090909091
0.7063210227272727
0.7063210227272727
0.7063210227272727
0.7077414772727273
0.7080965909090909
0.7080965909090909
0.7084517